# Advanced Data-Driven Modeling: Machine Learning
## Studi Kasus: Data Transmisi PLN — Proyek SUTT 150kV Kalimantan

**Mata Kuliah:** Metode Eksperimen  
**Program Studi:** S2 Manajemen Proyek Konstruksi  

---

### Tujuan Pembelajaran
1. Memahami pipeline machine learning end-to-end pada data proyek konstruksi
2. Mengaplikasikan Distribution Fitting dan Monte Carlo Simulation untuk risk analysis
3. Membangun model klasifikasi untuk memprediksi apakah suatu proyek akan mengalami *cost overrun*
4. Membangun model regresi untuk memprediksi besaran persentase *cost overrun*
5. Menginterpretasikan feature importance dan sensitivity analysis (Tornado Chart)

### Dataset
- **n = 34** proyek SUTT 150kV di Kalimantan (2010–2020)
- **Variabel target (klasifikasi):** Overrun Flag (1 = overrun, 0 = on-budget/saving)
- **Variabel target (regresi):** Deviasi Cost Overrun (%)

---

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, lognorm, weibull_min
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    mean_squared_error, r2_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
print('✅ Libraries loaded successfully')

## 1. Data Loading & Preprocessing

In [ ]:
# Helper functions to parse Indonesian currency format
def parse_rp(s):
    """Parse Rupiah string to float"""
    if pd.isna(s): return np.nan
    s = str(s).replace(' ','').replace('Rp','').replace(',','')
    neg = s.startswith('-'); s = s.lstrip('-')
    try: return -float(s) if neg else float(s)
    except: return np.nan

def parse_pct(s):
    """Parse percentage string to float"""
    if pd.isna(s): return np.nan
    return float(str(s).replace('%','').strip())

# Load raw data
raw = pd.read_csv('Data_Transmisi_PLN.csv')  # adjust path as needed
cols = raw.columns.tolist()

# Clean and rename columns
df = pd.DataFrame({
    'ID':             raw[cols[0]],
    'Provinsi':       raw[cols[2]].str.strip(),
    'Total_Tower':    pd.to_numeric(raw[cols[3]], errors='coerce'),
    'Tahun_Kontrak':  pd.to_numeric(raw[cols[4]], errors='coerce'),
    'Durasi':         pd.to_numeric(raw[cols[5]], errors='coerce'),
    'Nilai_Original': raw[cols[6]].apply(parse_rp),
    'Nilai_Akhir':    raw[cols[7]].apply(parse_rp),
    'Deviasi_Nilai':  raw[cols[8]].apply(parse_rp),
    'Kategori':       raw[cols[9]].str.strip(),
    'Deviasi_Pct':    raw[cols[10]].apply(parse_pct),
    'Jenis':          raw[cols[11]].str.strip(),
    'Harga_Per_Tower':raw[cols[12]].apply(parse_rp),
})

# Convert to Miliar IDR for readability
for c in ['Nilai_Original','Nilai_Akhir','Deviasi_Nilai']:
    df[c+'_B'] = df[c] / 1e9
df['Harga_Per_Tower_M'] = df['Harga_Per_Tower'] / 1e6

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Feature Engineering ─────────────────────────────────────────────────
df['Overrun_Flag']   = (df['Deviasi_Pct'] > 0).astype(int)  # Binary target
df['Overrun_Label']  = df['Overrun_Flag'].map({1:'Overrun', 0:'On-Budget/Saving'})
df['Tower_per_Day']  = df['Total_Tower'] / df['Durasi']       # Productivity proxy
df['Cost_Intensity'] = df['Nilai_Original_B'] / df['Total_Tower']  # Cost per tower

# Encode categorical variables
le = LabelEncoder()
df['Kat_enc']   = le.fit_transform(df['Kategori'].fillna('Unknown'))
df['Jenis_enc'] = le.fit_transform(df['Jenis'].fillna('Unknown'))
df['Prov_enc']  = le.fit_transform(df['Provinsi'].fillna('Unknown'))

# Drop rows with missing key features
df_clean = df.dropna(subset=['Total_Tower','Durasi','Nilai_Original_B',
                               'Deviasi_Pct','Harga_Per_Tower_M']).copy()

print(f'Clean dataset: {len(df_clean)} rows')
print(f"\nClass balance:")
print(df_clean['Overrun_Label'].value_counts())
print(f"\nOverrun rate: {df_clean['Overrun_Flag'].mean()*100:.1f}%")

In [ ]:
# ── Define Feature Sets & Prepare Data ─────────────────────────────────
FEATURES = ['Total_Tower','Durasi','Nilai_Original_B','Harga_Per_Tower_M',
            'Tower_per_Day','Cost_Intensity','Kat_enc','Jenis_enc','Tahun_Kontrak']

FEAT_LABELS = ['Total Tower','Durasi','Nilai Original (B)','Harga/Tower (M)',
               'Tower/Day','Cost Intensity','Kategori','Jenis','Tahun Kontrak']

X = df_clean[FEATURES].fillna(df_clean[FEATURES].median())
y_class = df_clean['Overrun_Flag']    # Classification target
y_reg   = df_clean['Deviasi_Pct']     # Regression target

# Standardize features
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

print('Feature matrix shape:', Xs.shape)
print('Features:', FEATURES)

## 2. Distribution Fitting & Monte Carlo Simulation

> **Tujuan:** Sebelum membangun model prediktif, kita perlu memahami distribusi probabilitas data. Ini adalah fondasi dari simulasi berbasis probabilitas.

### 2.1 Distribution Fitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, xlabel in [
    (axes[0], y_reg, 'Cost Overrun (%)', 'Deviasi (%)'),
    (axes[1], df_clean['Nilai_Akhir_B'], 'Nilai Kontrak Akhir', 'Rp Miliar'),
]:
    data_np = data.dropna().values
    ax.hist(data_np, bins=10, density=True, alpha=0.5, color='steelblue', label='Data')

    # Fit Normal distribution
    mu, sigma = norm.fit(data_np)
    xf = np.linspace(data_np.min()-2, data_np.max()+2, 300)
    ax.plot(xf, norm.pdf(xf, mu, sigma), 'r-', lw=2.5,
            label=f'Normal (μ={mu:.1f}, σ={sigma:.1f})')

    # Kolmogorov-Smirnov test
    ks_stat, p_val = stats.kstest(data_np, 'norm', args=(mu, sigma))
    ax.set_title(f'{title}\nKS test p={p_val:.3f} ({"fit OK" if p_val>0.05 else "poor fit"})')
    ax.set_xlabel(xlabel); ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Distribution Fitting', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nCost Overrun: Mean={y_reg.mean():.2f}%, Std={y_reg.std():.2f}%')
print(f'Nilai Kontrak: Mean=Rp{df_clean["Nilai_Akhir_B"].mean():.1f}B, Std=Rp{df_clean["Nilai_Akhir_B"].std():.1f}B')

### 2.2 Monte Carlo Simulation

> Mensimulasikan **10.000 skenario** nilai kontrak akhir berdasarkan distribusi historis cost overrun.

In [ ]:
np.random.seed(42)
N_SIM = 10_000

# Parameters from historical data
mu_overrun  = y_reg.mean()
std_overrun = y_reg.std()
mu_nilai    = df_clean['Nilai_Original_B'].mean()
std_nilai   = df_clean['Nilai_Original_B'].std()

# Simulate
sim_overrun = np.random.normal(mu_overrun, std_overrun, N_SIM)
sim_nilai   = np.random.normal(mu_nilai, std_nilai, N_SIM)
sim_final   = sim_nilai * (1 + sim_overrun / 100)

# Risk percentiles
p10, p50, p90 = np.percentile(sim_final, [10, 50, 90])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of simulated outcomes
axes[0].hist(sim_final, bins=60, color='steelblue', alpha=0.65, density=True)
for pv, pc, col, lbl in [(p10,10,'green','P10'),(p50,50,'orange','P50'),(p90,90,'red','P90')]:
    axes[0].axvline(pv, color=col, lw=2, ls='--', label=f'{lbl} = Rp{pv:.1f}B')
axes[0].set_title(f'Monte Carlo: {N_SIM:,} Simulated Contract Values')
axes[0].set_xlabel('Nilai Kontrak Akhir (Rp Miliar)'); axes[0].set_ylabel('Density')
axes[0].legend()

# P-Curve (Cumulative Risk Profile)
percs = np.arange(1, 100)
perc_vals = np.percentile(sim_final, percs)
axes[1].plot(percs, perc_vals, 'steelblue', lw=2.5)
axes[1].fill_between(percs, perc_vals, alpha=0.1, color='steelblue')
for pv, pc, col in [(p10,10,'green'),(p50,50,'orange'),(p90,90,'red')]:
    axes[1].scatter([pc], [pv], color=col, s=80, zorder=5)
    axes[1].annotate(f'P{pc}=Rp{pv:.1f}B', (pc,pv), xytext=(5,5),
                     textcoords='offset points', color=col, fontweight='bold', fontsize=9)
axes[1].set_title('Cumulative Risk Profile (P-Curve)')
axes[1].set_xlabel('Percentile'); axes[1].set_ylabel('Nilai Kontrak Akhir (Rp Miliar)')

plt.suptitle('Monte Carlo Simulation — Cost Risk Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📊 Risk Summary:')
print(f'  P10 (optimistic) : Rp{p10:.1f} Miliar')
print(f'  P50 (most likely): Rp{p50:.1f} Miliar')
print(f'  P90 (worst-case) : Rp{p90:.1f} Miliar')
print(f'  Swing P10→P90    : Rp{p90-p10:.1f} Miliar ({(p90-p10)/p50*100:.1f}% of P50)')

## 3. Classification Models — Predicting Cost Overrun (Yes/No)

> **Pertanyaan:** *Dapatkah kita memprediksi apakah sebuah proyek baru akan mengalami cost overrun sebelum kontrak selesai?*

### 3.1 Model Training & Cross-Validation

Karena n=33, kita gunakan **5-Fold Stratified Cross-Validation** bukan simple train-test split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=42),
}

print('5-Fold Cross-Validation Results:')
print('-' * 50)
clf_scores = {}
for name, clf in clf_models.items():
    scores = cross_val_score(clf, Xs, y_class, cv=cv, scoring='accuracy')
    clf_scores[name] = scores
    print(f'{name:25s}: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
# Visualization: CV Scores + ROC Curves
X_tr, X_te, y_tr, y_te = train_test_split(Xs, y_class, test_size=0.3,
                                            random_state=42, stratify=y_class)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV Accuracy comparison
short_names = ['Logistic\nReg.','Decision\nTree','Random\nForest','Grad.\nBoost']
means = [clf_scores[n].mean() for n in clf_models]
stds  = [clf_scores[n].std()  for n in clf_models]
bars = axes[0].bar(short_names, means, yerr=stds, capsize=5,
                    color=['#1F5C99','#E8772E','#2EAE8F','#7B52AB'], edgecolor='white')
axes[0].axhline(0.5, color='gray', ls='--', lw=1.5, label='Random baseline')
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel('Accuracy')
axes[0].set_title('5-Fold CV Accuracy')
for b, m in zip(bars, means):
    axes[0].text(b.get_x()+b.get_width()/2, m+0.04, f'{m:.2f}',
                 ha='center', fontweight='bold', fontsize=9)
axes[0].legend()

# ROC Curves
colors_roc = ['#1F5C99','#E8772E','#2EAE8F','#7B52AB']
for (name, clf), col in zip(clf_models.items(), colors_roc):
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:,1] if hasattr(clf,'predict_proba') else clf.decision_function(X_te)
    fpr, tpr, _ = roc_curve(y_te, proba)
    axes[1].plot(fpr, tpr, lw=2, color=col, label=f'{name.split()[0]} (AUC={auc(fpr,tpr):.2f})')
axes[1].plot([0,1],[0,1],'--',color='gray',label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves'); axes[1].legend(fontsize=8)

plt.suptitle('Classification Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Decision Tree Visualization

> Decision Tree adalah model yang paling **interpretable** — kita bisa melihat persis aturan keputusan yang digunakan.

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(Xs, y_class)

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(dt, feature_names=FEAT_LABELS, class_names=['On-Budget','Overrun'],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title('Decision Tree — Cost Overrun Classification (max_depth=3)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print text rules
from sklearn.tree import export_text
print('\nDecision Rules:')
print(export_text(dt, feature_names=FEAT_LABELS))

### 3.3 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, clf) in zip(axes, {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=42),
}.items()):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['On-Budget','Overrun']).plot(ax=ax, cmap='Blues', colorbar=False)
    acc = cm.diagonal().sum() / cm.sum()
    ax.set_title(f'{name}\nAcc={acc:.2f}')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Regression Models — Predicting Cost Overrun (%)

> **Pertanyaan:** *Berapa persen cost overrun yang dapat kita prediksi untuk proyek baru?*

In [ ]:
cv_kf = KFold(n_splits=5, shuffle=True, random_state=42)

reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Decision Tree':     DecisionTreeRegressor(max_depth=3, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42),
}

print('5-Fold Cross-Validation Results (Regression):')
print('-' * 60)
reg_scores = {}
for name, reg in reg_models.items():
    r2   = cross_val_score(reg, Xs, y_reg, cv=cv_kf, scoring='r2')
    rmse = cross_val_score(reg, Xs, y_reg, cv=cv_kf, scoring='neg_root_mean_squared_error')
    reg_scores[name] = {'r2': r2, 'rmse': -rmse}
    print(f'{name:22s}: R²={r2.mean():.3f}±{r2.std():.3f}  RMSE={(-rmse).mean():.2f}±{(-rmse).std():.2f}%')

In [ ]:
# Actual vs Predicted — Random Forest Regressor
scaler_r = StandardScaler()
Xsr = scaler_r.fit_transform(X)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xsr, y_reg, test_size=0.3, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42)
rf_reg.fit(Xr_tr, yr_tr)
yr_pred = rf_reg.predict(Xr_te)

r2_test   = r2_score(yr_te, yr_pred)
rmse_test = np.sqrt(mean_squared_error(yr_te, yr_pred))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# R² comparison
names_short = ['Linear','Ridge','Dec. Tree','Rand. Forest']
r2_means = [reg_scores[n]['r2'].mean() for n in reg_models]
axes[0].bar(names_short, r2_means, color=['#1F5C99','#E8772E','#2EAE8F','#7B52AB'], edgecolor='white')
axes[0].axhline(0, color='gray', ls='--', lw=1)
axes[0].set_title('5-Fold CV R² Score'); axes[0].set_ylabel('R²')
for i, v in enumerate(r2_means):
    axes[0].text(i, v+0.02, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

# Actual vs Predicted
axes[1].scatter(yr_te, yr_pred, color='#1F5C99', s=80, alpha=0.85, edgecolors='white')
lim = [min(yr_te.min(), yr_pred.min())-1, max(yr_te.max(), yr_pred.max())+1]
axes[1].plot(lim, lim, 'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Deviasi (%)'); axes[1].set_ylabel('Predicted Deviasi (%)')
axes[1].set_title(f'Random Forest: Actual vs Predicted\nR²={r2_test:.2f}, RMSE={rmse_test:.2f}%')
axes[1].legend()

plt.suptitle('Regression Model Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Importance & Sensitivity Analysis

In [ ]:
# Random Forest feature importance
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42)
rf_clf.fit(Xs, y_class)

importances = rf_clf.feature_importances_
perm = permutation_importance(rf_clf, Xs, y_class, n_repeats=30, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gini importance
idx = np.argsort(importances)
axes[0].barh([FEAT_LABELS[i] for i in idx], importances[idx],
             color='#1F5C99', edgecolor='white', alpha=0.85)
axes[0].set_title('Gini Importance (Random Forest)'); axes[0].set_xlabel('Score')

# Permutation importance (Tornado)
pidx = np.argsort(perm.importances_mean)
colors = ['#C0392B' if v > 0 else '#666666' for v in perm.importances_mean[pidx]]
axes[1].barh([FEAT_LABELS[i] for i in pidx], perm.importances_mean[pidx],
             xerr=perm.importances_std[pidx], color=colors,
             edgecolor='white', capsize=4)
axes[1].axvline(0, color='black', lw=1.2)
axes[1].set_title('Permutation Importance (Tornado)'); axes[1].set_xlabel('Mean Accuracy Decrease')

plt.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nTop 3 most important features:')
top3 = np.argsort(importances)[::-1][:3]
for rank, i in enumerate(top3, 1):
    print(f'  {rank}. {FEAT_LABELS[i]}: {importances[i]:.3f}')

### 5.1 Sensitivity Analysis — Tornado Chart

In [ ]:
# Vary each feature ±1 std, record impact on predicted cost overrun
rf_reg2 = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42)
rf_reg2.fit(Xsr, y_reg)

baseline_input = X.median().values.reshape(1, -1)
baseline_scaled = scaler_r.transform(baseline_input)
baseline_pred   = rf_reg2.predict(baseline_scaled)[0]

low_impacts, high_impacts = [], []
for i, feat in enumerate(X.columns):
    lo = baseline_input.copy(); lo[0, i] -= X[feat].std()
    hi = baseline_input.copy(); hi[0, i] += X[feat].std()
    low_impacts.append(rf_reg2.predict(scaler_r.transform(lo))[0] - baseline_pred)
    high_impacts.append(rf_reg2.predict(scaler_r.transform(hi))[0] - baseline_pred)

swing = [abs(h-l) for h,l in zip(high_impacts, low_impacts)]
sort_idx = np.argsort(swing)

fig, ax = plt.subplots(figsize=(11, 5))
for yi, i in enumerate(sort_idx):
    ax.barh(yi, low_impacts[i],  left=baseline_pred, color='#27AE60', alpha=0.85, height=0.6)
    ax.barh(yi, high_impacts[i], left=baseline_pred, color='#C0392B', alpha=0.85, height=0.6)
ax.set_yticks(range(len(sort_idx)))
ax.set_yticklabels([FEAT_LABELS[i] for i in sort_idx])
ax.axvline(baseline_pred, color='black', lw=2, label=f'Baseline = {baseline_pred:.1f}%')
ax.set_xlabel('Predicted Cost Overrun (%)')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#C0392B', alpha=0.85, label='+1 Std (increase overrun)'),
    Patch(color='#27AE60', alpha=0.85, label='−1 Std (decrease overrun)'),
    plt.Line2D([0],[0], color='black', lw=2, label=f'Baseline={baseline_pred:.1f}%')
], fontsize=9)

plt.title(f'Sensitivity Analysis — Tornado Chart\n(Impact of ±1 Std on Predicted Cost Overrun)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBaseline prediction: {baseline_pred:.2f}%')
print('\nTop 3 most sensitive features:')
for i in sort_idx[-3:][::-1]:
    print(f'  {FEAT_LABELS[i]}: swing = {swing[i]:.2f}%')

## 6. Model Summary & Interpretasi

> Bagian ini merangkum semua hasil dan memberikan interpretasi praktis untuk manajemen proyek konstruksi.

In [ ]:
print('=' * 65)
print('  MODEL PERFORMANCE SUMMARY')
print('=' * 65)

print('\n📊 CLASSIFICATION (Predict: Overrun Yes/No)')
print('-' * 40)
for name in clf_models:
    sc = clf_scores[name]
    print(f'  {name:25s}: Acc = {sc.mean():.2f} ± {sc.std():.2f}')

print('\n📈 REGRESSION (Predict: Overrun %)')
print('-' * 40)
for name in reg_models:
    r2   = reg_scores[name]['r2']
    rmse = reg_scores[name]['rmse']
    print(f'  {name:22s}: R² = {r2.mean():.2f} ± {r2.std():.2f}  |  RMSE = {rmse.mean():.2f}%')

print('\n🎲 MONTE CARLO RISK SUMMARY (10,000 simulations)')
print('-' * 40)
print(f'  P10 (Best Case)  : Rp {p10:.1f} Miliar')
print(f'  P50 (Most Likely): Rp {p50:.1f} Miliar')
print(f'  P90 (Worst Case) : Rp {p90:.1f} Miliar')

print('\n⚠️  CATATAN PENTING:')
print('  • n=33 sangat kecil untuk ML — hasil ini bersifat ILUSTRATIF')
print('  • Gunakan dengan hati-hati untuk generalisasi di luar dataset ini')
print('  • Model terbaik butuh data lebih banyak & fitur tambahan')
print('=' * 65)

---

## Catatan Metodologis

| Aspek | Nilai | Catatan |
|-------|-------|---------|
| Sample size | n = 33 | Sangat kecil untuk ML — hasil illustratif |
| Class imbalance | 73% Overrun vs 27% On-Budget | Bias ke majority class |
| Cross-validation | 5-Fold Stratified | Lebih robust dari single train-test split |
| Feature scaling | StandardScaler | Z-score normalization |
| Best classifier | Decision Tree (acc≈0.72) | Paling interpretable |
| Best regressor | Random Forest (R²≈0.25) | R² rendah — overrun sulit diprediksi |

### Saran Pengembangan
1. **Perbesar dataset** — idealnya n > 100 untuk ML yang reliable
2. **Tambah fitur** — kondisi cuaca, kompleksitas medan, kontraktor, dll.
3. **Coba SMOTE** untuk menangani class imbalance
4. **Bayesian Optimization** untuk hyperparameter tuning
5. **Ensemble stacking** untuk meningkatkan performa prediksi